# Lab 24-07 · TIA, Dispersion, Modulators and Sensitivity

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import optics


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · TIA

*Predict* the transimpedance and bandwidth.

In [2]:
for rf in (1e3, 2e3):
    z, f3 = optics.tia(rf, 10, 100e-15)
    print(f"R_F {rf/1e3:.0f} kohm: Z_T {z:.0f} ohm, bandwidth {f3/1e9:.2f} GHz, "
          f"resistor noise {optics.resistor_noise_a_rthz(rf)*1e12:.2f} pA/rtHz")
sim = ngspice("tia.cir")
print(f"ngspice (1 kohm): Z_T {sim['zlow']:.1f} ohm, bandwidth {sim['f3db']/1e9:.2f} GHz")

R_F 1 kohm: Z_T 909 ohm, bandwidth 17.51 GHz, resistor noise 4.07 pA/rtHz
R_F 2 kohm: Z_T 1818 ohm, bandwidth 8.75 GHz, resistor noise 2.88 pA/rtHz
ngspice (1 kohm): Z_T 909.1 ohm, bandwidth 17.51 GHz


## Part 2 · Dispersion

*Predict* the broadening of a 5 ps Gaussian over 2 km at 1550 and 1310 nm.

In [3]:
plt.figure()
for d, lam, lab in ((18, 1550e-9, "1550 nm, D = 18"), (3.5, 1310e-9, "1310 nm, D = 3.5")):
    t, p0, p1 = optics.propagate_gaussian(5.0, d, 2.0, lam)
    b2 = d * (lam * 1e9) ** 2 / (2 * math.pi * optics.C0 * 1e-3)
    ld = 25.0 / b2
    print(f"{lab}: |beta2| {b2:.2f} ps^2/km, L_D {ld:.2f} km, FFT broadening "
          f"{optics.rms_width(t, p1)/optics.rms_width(t, p0):.4f}, analytic {math.sqrt(1 + (2/ld)**2):.4f}")
    plt.plot(t, p1, label=f"after 2 km, {lab}")
plt.plot(t, p0, "k--", label="launched")
plt.xlim(-40, 40); plt.xlabel("time (ps)"); plt.ylabel("power (normalized)"); plt.legend(); plt.grid(True)
plt.show()

1550 nm, D = 18: |beta2| 22.96 ps^2/km, L_D 1.09 km, FFT broadening 2.0912, analytic 2.0912
1310 nm, D = 3.5: |beta2| 3.19 ps^2/km, L_D 7.84 km, FFT broadening 1.0320, analytic 1.0320


## Part 3 · Modulators

*Predict* the MZM extinction ratio with a half-Vpi drive and the ring's shift for 2.3 K.

In [4]:
hi, lo = optics.mzm(math.pi / 2 - math.pi / 4), optics.mzm(math.pi / 2 + math.pi / 4)
print(f"MZM: high {hi:.3f}, low {lo:.3f}, ER {10*math.log10(hi/lo):.2f} dB")
lam = np.linspace(1309.7, 1310.4, 800)
shift = 1310 * 1.8e-4 / 4.2
plt.figure()
for dt in (0, 1, 2.3):
    plt.plot(lam, optics.ring_transmission(lam, 1310 + shift * dt, 1e4), label=f"+{dt} K")
print(f"ring shift for 2.3 K: {shift*2.3:.3f} nm (linewidth {1310/1e4:.3f} nm)")
plt.xlabel("wavelength (nm)"); plt.ylabel("transmission"); plt.legend(); plt.grid(True)
plt.show()

MZM: high 0.854, low 0.146, ER 7.66 dB
ring shift for 2.3 K: 0.129 nm (linewidth 0.131 nm)


## Part 4 · Sensitivity

*Predict* the power for 10⁻¹² with OOK and PAM4.

In [5]:
r, sigma = 0.9, 20e-12 * math.sqrt(30e9)
pdbm = np.linspace(-22, -6, 200)
pw = 1e-3 * 10 ** (pdbm / 10)
plt.figure()
plt.semilogy(pdbm, optics.ber_ook(pw, r, sigma), label="OOK")
plt.semilogy(pdbm, optics.ber_pam4(pw, r, sigma), label="PAM4")
for target in (1e-12, 2.4e-4):
    plt.axhline(target, color="0.6", ls=":")
for name, fn in (("OOK", optics.ber_ook), ("PAM4", optics.ber_pam4)):
    p = optimize.brentq(lambda x: math.log(max(float(fn(1e-3 * 10 ** (x / 10), r, sigma)), 1e-300)) + 12 * math.log(10), -25, -5)
    print(f"{name}: 1e-12 at {p:.2f} dBm")
plt.ylim(1e-15, 1); plt.xlabel("average received power (dBm)"); plt.ylabel("BER"); plt.legend(); plt.grid(True, which="both")
plt.show()

OOK: 1e-12 at -15.67 dBm
PAM4: 1e-12 at -10.93 dBm
